## Demo 1: QuickStart with PySpark in Jupyter Notebooks
This exercise will assist students with configuring their environment to run PySpark. Configurations will differ slightly between computers running Microsoft Windows and those running the MacOS.  Because Spark and Hadoop were both originally developed for UNIX operating systems (e.g., Ubuntu, Linux), configuration is slightly simpler for the, UNIX-based, MacIntosh OS. There are a few extra steps required to run Spark on the Windows platform so that it can interact with the underlying Hadoop file system (HDFS) in order to save data and schema definitions.

#### Overall, the steps are as follows:
- Confirm that <a href="https://www.anaconda.com/download/success"><b>Anaconda Python with Jupyter Notebooks</b></a> is already installed on your computer.
    - Create a new Conda Environment that uses Python version 3.12.7 and the Anaconda libraries. `conda create -n pysparkenv python==3.12.7 anaconda`
    - Activate the new Conda Environment. `conda activate pysparkenv`
    - Install the Jupyter library in the new environment. `python -m pip install ipykernel`
    - Make the Jupyter kernel available. `python -m ipykernel install --user --name pysparkenv --display-name "Python 3 (pysparkenv)"`
- Download and install the <a href="https://www.oracle.com/java/technologies/downloads/"><b>Java 21 runtime</b></a>.
    - (Windows ONLY) Ensure you change the default installation path from `"C:\Program Files\Java\jdk-21"` to `"C:\Java\jdk-21"`.
- **Windows ONLY:**
    - Download <a href="https://www.apache.org/dyn/closer.lua/spark/spark-3.5.4/spark-3.5.4-bin-hadoop3.tgz"><b>Apache Spark release 3.4.5 (Dec 20 2024)</b></a>, package type <b>Pre-built for Apache Hadoop 3.3 and later</b>. Copy it to `C:\spark-4.5.4-bin-hadoop3`.
    - Download <a href="https://github.com/cdarlint/winutils"><b>Winutils Hadoop-3.3.6</b></a> and copy it to `C:\hadoop-3.3.6`. 
    - Configure Local User Environmental Variables:
      - <b>JAVA_HOME</b> that points to the `C:\Java\jdk-21` directory.
      - <b>SPARK_HOME</b> that points to the `C:\spark-3.5.4-bin-hadoop3` directory.
      - <b>HADOOP_HOME</b> that points to the `C:\hadoop-3.3.6` directory.
  - Append the `%JAVA_HOME%\bin`, `%SPARK_HOME%\bin` and `%HADOOP_HOME%\bin` paths to the <b>Path</b> variable without overwriting any of the existing entries.
- **Mac ONLY** Use the Python Installer Program (PIP) to install <b>Spark</b> and <b>PySpark</b> in your Anaconda Python Environment.
  - `python -m pip install spark`
  - `python -m pip install pyspark`
- Use the Python Installer Program (PIP) to install <b>findpark</b> in your Anaconda Python environment; `python -m pip install findspark`.
- Use the Python Installer Program (PIP) to install <b>Delta Lake</b> support in your Anaconda Python environment; `python -m pip install delta-spark==3.3.0`

#### Import Required Libraries

In [1]:
import findspark
findspark.init()
findspark.find()

'/opt/anaconda3/envs/pysparkenv/lib/python3.12/site-packages/pyspark'

In [2]:
import os
import sys
import shutil

from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

#### Instantiate Global Variables

In [3]:
# --------------------------------------------------------------------------------
# Specify Directory Structure for Source Data
# --------------------------------------------------------------------------------
base_dir = os.path.join(os.getcwd(), 'lab_data')
data_dir = os.path.join(base_dir, 'retail-org')
customers_dir = os.path.join(data_dir, "customers")

# --------------------------------------------------------------------------------
# Create Directory Structure for Data Lakehouse Files
# --------------------------------------------------------------------------------
dest_database = "quickstart"
sql_warehouse_dir = os.path.abspath('spark-warehouse')
dest_database_dir = f"{dest_database}.db"
database_dir = os.path.join(sql_warehouse_dir, dest_database_dir)

#### Define Utilities

In [4]:
def remove_directory_tree(path: str):
    '''If it exists, remove the entire contents of a directory structure at a given 'path' parameter's location.'''
    try:
        if os.path.exists(path):
            shutil.rmtree(path)
            return f"Directory '{path}' has been removed successfully."
        else:
            return f"Directory '{path}' does not exist."
            
    except Exception as e:
        return f"An error occurred: {e}"

#### Create a New Spark Session

In [5]:
worker_threads = f"local[{int(os.cpu_count()/2)}]"
shuffle_partitions = int(os.cpu_count())

sparkConf = SparkConf().setAppName('PySpark Quickstart in Juptyer')\
    .setMaster(worker_threads)\
    .set('spark.executor.memory', '2g')\
    .set('spark.driver.memory', '4g') \
    .set('spark.sql.shuffle.partitions', shuffle_partitions) \
    .set('spark.sql.warehouse.dir', sql_warehouse_dir)

spark = SparkSession.builder.config(conf=sparkConf).getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/01 09:20:45 WARN Utils: Your hostname, Tianyaos-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 172.25.162.150 instead (on interface en0)
26/04/01 09:20:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/01 09:20:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/01 09:20:47 WARN FileSystem: Cannot load filesystem
java.util.ServiceConfigurationError: org.apache.hadoop.fs.FileSystem: Provider org.apache.hadoop.fs.viewfs.ViewFileSystem could not be instantiated
	at java.base/java.util.ServiceLoader.fail(ServiceLoader.java:552)
	at java.base/java.util.ServiceLoader$Pr

#### Prepare Filesystem

In [ ]:
remove_directory_tree(database_dir)

#### Read Customer data from a CSV File

In [ ]:
customers_csv = os.path.join(customers_dir, "customers.csv")
print(customers_csv)

In [ ]:
df_customers = spark.read.format('csv').options(header='true', inferSchema=True).load(customers_csv)

# Unit Test -------------
print(f"The 'df_customers' object is of type: {type(df_customers)}.")
df_customers.printSchema()

print(f"The 'df_customers' DataFrame contains {df_customers.count()} rows.")
df_customers.toPandas().head(5)

#### Persist the 'df_customers' DataFrame as a New Table in the Data Lakehouse
##### Create a New Data Lakehouse Database

In [ ]:
spark.sql(f"DROP DATABASE IF EXISTS {dest_database};")
spark.sql(f"CREATE DATABASE {dest_database};")

##### Create the 'customers' table.

In [ ]:
df_customers.write.saveAsTable(f"{dest_database}.customers", mode="overwrite")

In [ ]:
# Unit Test ------------------------------------
sql_customers = f"""
    SELECT customer_id
        , customer_name
        , CONCAT(number, " ", street) AS address
        , city
        , state
        , postcode
        , FLOOR(units_purchased) AS units_purchased
    FROM {dest_database}.customers
    ORDER BY state ASC;
"""

spark.sql(sql_customers).toPandas().head(5)

#### Who are my best customers? (i.e., Which customers purchased the most product?)

In [ ]:
sql_best_customers = f"""
    SELECT customer_name AS Customer
        , FLOOR(SUM(units_purchased)) AS Total_Units_Purchased
    FROM {dest_database}.customers
    GROUP BY customer_name
    HAVING total_units_purchased >= 750
    ORDER BY total_units_purchased DESC;
"""

spark.sql(sql_best_customers).toPandas()

In [ ]:
spark.stop()